# Hex Base Game

In [1]:
import numpy as np

class HexState:
    def __init__(self, board_size=11, board=None, turn=1):
        """
        Represents the state of a Hex game.
        
        Player 1 (Vertical): Tries to connect top row (row 0) to bottom row (size-1).
        Player 2 (Horizontal): Tries to connect left column (col 0) to right column (size-1).
        """
        self.board_size = board_size
        self.turn = turn  # 1 or 2
        
        if board is None:
            self.board = np.zeros((board_size, board_size), dtype=int)
        else:
            self.board = np.copy(board)
            
        self._cached_winner = None
        self._is_terminal_cached = False

    def get_legal_moves(self):
        """Returns a list of tuples (row, col) representing empty spaces."""
        if self.is_terminal():
            return []
        # Non-zero entries are occupied
        rows, cols = np.where(self.board == 0)
        return list(zip(rows, cols))

    def make_move(self, move):
        """
        Returns a NEW HexState instance representing the state after the move.
        Crucial for MCTS to avoid modifying historical board states during simulation.
        """
        row, col = move
        if self.board[row, col] != 0:
            raise ValueError(f"Move {move} is invalid. Cell already occupied.")
            
        next_board = np.copy(self.board)
        next_board[row, col] = self.turn
        next_turn = 3 - self.turn  # Swaps between 1 and 2
        
        return HexState(self.board_size, next_board, next_turn)

    def get_neighbors(self, row, col):
        """Returns the up to 6 valid neighboring coordinates on a Hex grid."""
        neighbors = []
        # The 6 standard hex directions
        directions = [
            (row - 1, col), (row - 1, col + 1),
            (row, col - 1), (row, col + 1),
            (row + 1, col - 1), (row + 1, col)
        ]
        for r, c in directions:
            if 0 <= r < self.board_size and 0 <= c < self.board_size:
                neighbors.append((r, c))
        return neighbors

    def check_winner(self):
        """
        Determines the winner of the game using an efficient BFS pathfinder.
        Hex cannot end in a draw, so if the board is full, one player must have won.
        """
        if self._cached_winner is not None:
            return self._cached_winner

        # Check Player 1: Top to Bottom connection
        if self._has_connected_path(player=1):
            self._cached_winner = 1
            return 1
            
        # Check Player 2: Left to Right connection
        if self._has_connected_path(player=2):
            self._cached_winner = 2
            return 2

        return None

    def _has_connected_path(self, player):
        """BFS helper to check connectivity from border to border for a given player."""
        queue = []
        visited = set()

        if player == 1:
            # Player 1 starts at any cell in the top row (row 0) owned by Player 1
            for c in range(self.board_size):
                if self.board[0, c] == 1:
                    queue.append((0, c))
                    visited.add((0, c))
        else:
            # Player 2 starts at any cell in the left column (col 0) owned by Player 2
            for r in range(self.board_size):
                if self.board[r, 0] == 2:
                    queue.append((r, 0))
                    visited.add((r, 0))

        while queue:
            curr_r, curr_c = queue.pop(0)

            # Check target boundary conditions
            if player == 1 and curr_r == self.board_size - 1:
                return True
            if player == 2 and curr_c == self.board_size - 1:
                return True

            for nr, nc in self.get_neighbors(curr_r, curr_c):
                if self.board[nr, nc] == player and (nr, nc) not in visited:
                    visited.add((nr, nc))
                    queue.append((nr, nc))

        return False

    def is_terminal(self):
        """Returns True if a player has won or if the board is full."""
        if self._is_terminal_cached:
            return True
            
        winner = self.check_winner()
        if winner is not None:
            self._is_terminal_cached = True
            return True
            
        # If no empty slots remain, the game is terminal
        if not np.any(self.board == 0):
            self._is_terminal_cached = True
            return True
            
        return False

    def __str__(self):
        """Generates a text-based representation of the rhomboid Hex board."""
        symbols = {0: '.', 1: 'X', 2: 'O'}
        lines = []
        for r in range(self.board_size):
            indent = " " * r
            row_str = " ".join(symbols[self.board[r, c]] for c in range(self.board_size))
            lines.append(f"{indent}{row_str}")
        return "\n".join(lines)

# Running the Base Hex Game

In [2]:
if __name__ == "__main__":
    # 1. Initialize a 4x4 test environment to keep validation execution swift
    print("Initializing a 4x4 Hex Board...")
    state = HexState(board_size=4)
    print(state)
    print("-" * 20)

    # 2. Simulate random play until a terminal state is reached
    import random
    
    step = 1
    while not state.is_terminal():
        moves = state.get_legal_moves()
        chosen_move = random.choice(moves)
        
        print(f"Step {step}: Player {state.turn} plays {chosen_move}")
        state = state.make_move(chosen_move)
        print(state)
        print("-" * 20)
        step += 1

    # 3. Output results
    winner = state.check_winner()
    print(f"Game Over! Winner: Player {winner} ({'X' if winner == 1 else 'O'})")

Initializing a 4x4 Hex Board...
. . . .
 . . . .
  . . . .
   . . . .
--------------------
Step 1: Player 1 plays (np.int64(2), np.int64(3))
. . . .
 . . . .
  . . . X
   . . . .
--------------------
Step 2: Player 2 plays (np.int64(2), np.int64(1))
. . . .
 . . . .
  . O . X
   . . . .
--------------------
Step 3: Player 1 plays (np.int64(1), np.int64(2))
. . . .
 . . X .
  . O . X
   . . . .
--------------------
Step 4: Player 2 plays (np.int64(0), np.int64(1))
. O . .
 . . X .
  . O . X
   . . . .
--------------------
Step 5: Player 1 plays (np.int64(0), np.int64(0))
X O . .
 . . X .
  . O . X
   . . . .
--------------------
Step 6: Player 2 plays (np.int64(0), np.int64(2))
X O O .
 . . X .
  . O . X
   . . . .
--------------------
Step 7: Player 1 plays (np.int64(1), np.int64(3))
X O O .
 . . X X
  . O . X
   . . . .
--------------------
Step 8: Player 2 plays (np.int64(3), np.int64(1))
X O O .
 . . X X
  . O . X
   . O . .
--------------------
Step 9: Player 1 plays (np.int64(2), 

# Base Gomoku Game

In [3]:
import numpy as np

class GomokuState:
    def __init__(self, board_size=15, board=None, turn=1, last_move=None):
        """
        Represents the state of a Gomoku game.
        
        Player 1: Plays 'X' (Black) - usually goes first.
        Player 2: Plays 'O' (White).
        Win Condition: Exactly or at least 5 consecutive pieces in a row 
                       (horizontal, vertical, or diagonal).
        """
        self.board_size = board_size
        self.turn = turn  # 1 or 2
        self.last_move = last_move  # Tracks the last played (row, col) for optimized win checking
        
        if board is None:
            self.board = np.zeros((board_size, board_size), dtype=int)
        else:
            self.board = np.copy(board)
            
        self._cached_winner = None
        self._is_terminal_cached = False

    def get_legal_moves(self):
        """Returns a list of tuples (row, col) representing empty spaces."""
        if self.is_terminal():
            return []
        rows, cols = np.where(self.board == 0)
        return list(zip(rows, cols))

    def make_move(self, move):
        """
        Returns a NEW GomokuState instance representing the state after the move.
        Ensures thread-safe/branch-safe execution during MCTS tree expansions.
        """
        row, col = move
        if self.board[row, col] != 0:
            raise ValueError(f"Move {move} is invalid. Cell already occupied.")
            
        next_board = np.copy(self.board)
        next_board[row, col] = self.turn
        next_turn = 3 - self.turn  # Swaps between 1 and 2
        
        return GomokuState(self.board_size, next_board, next_turn, last_move=move)

    def check_winner(self):
        """
        Determines the winner of the game.
        Optimized to scan outwards from the last played move across 4 directional axes.
        """
        if self._cached_winner is not None:
            return self._cached_winner

        if self.last_move is None:
            return None

        r, c = self.last_move
        player = self.board[r, c]  # The player who just made the move

        # The 4 directions to check: Horizontal, Vertical, Diagonal Down-Right, Diagonal Up-Right
        directions = [(0, 1), (1, 0), (1, 1), (1, -1)]

        for dr, dc in directions:
            count = 1  # Include the piece just placed
            
            # Search forward along the direction vector
            step = 1
            while True:
                nr, nc = r + dr * step, c + dc * step
                if 0 <= nr < self.board_size and 0 <= nc < self.board_size and self.board[nr, nc] == player:
                    count += 1
                    step += 1
                else:
                    break
                    
            # Search backward along the direction vector
            step = 1
            while True:
                nr, nc = r - dr * step, c - dc * step
                if 0 <= nr < self.board_size and 0 <= nc < self.board_size and self.board[nr, nc] == player:
                    count += 1
                    step += 1
                else:
                    break

            # Gomoku Win Check (Standard rule: 5 or more in a row wins)
            if count >= 5:
                self._cached_winner = player
                return player

        return None

    def is_terminal(self):
        """Returns True if a player has achieved 5-in-a-row or if the board is completely full (Draw)."""
        if self._is_terminal_cached:
            return True
            
        winner = self.check_winner()
        if winner is not None:
            self._is_terminal_cached = True
            return True
            
        # If no empty slots remain, it's a draw terminal state
        if not np.any(self.board == 0):
            self._is_terminal_cached = True
            return True
            
        return False

    def __str__(self):
        """Generates a text-based grid representation of the Gomoku board."""
        symbols = {0: '.', 1: 'X', 2: 'O'}
        lines = []
        
        # Add column headers for easier debugging
        header = "   " + " ".join(f"{c:02d}"[-2:] for c in range(self.board_size))
        lines.append(header)
        
        for r in range(self.board_size):
            row_str = " ".join(symbols[self.board[r, c]] for c in range(self.board_size))
            lines.append(f"{r:02d} {row_str}")
            
        return "\n".join(lines)

# Running the Base Gomoku Game

In [4]:
if __name__ == "__main__":
    # 1. Initialize a 6x6 test environment for fast evaluation
    print("Initializing a 6x6 Gomoku Board...")
    state = GomokuState(board_size=6)
    print(state)
    print("-" * 30)

    # 2. Simulate random play until a terminal state is reached
    import random
    
    step = 1
    while not state.is_terminal():
        moves = state.get_legal_moves()
        chosen_move = random.choice(moves)
        
        print(f"Step {step}: Player {state.turn} plays {chosen_move}")
        state = state.make_move(chosen_move)
        print(state)
        print("-" * 30)
        step += 1

    # 3. Output results
    winner = state.check_winner()
    if winner:
        print(f"Game Over! Winner: Player {winner} ({'X' if winner == 1 else 'O'})")
    else:
        print("Game Over! It's a Draw.")

Initializing a 6x6 Gomoku Board...
   00 01 02 03 04 05
00 . . . . . .
01 . . . . . .
02 . . . . . .
03 . . . . . .
04 . . . . . .
05 . . . . . .
------------------------------
Step 1: Player 1 plays (np.int64(4), np.int64(4))
   00 01 02 03 04 05
00 . . . . . .
01 . . . . . .
02 . . . . . .
03 . . . . . .
04 . . . . X .
05 . . . . . .
------------------------------
Step 2: Player 2 plays (np.int64(1), np.int64(4))
   00 01 02 03 04 05
00 . . . . . .
01 . . . . O .
02 . . . . . .
03 . . . . . .
04 . . . . X .
05 . . . . . .
------------------------------
Step 3: Player 1 plays (np.int64(3), np.int64(0))
   00 01 02 03 04 05
00 . . . . . .
01 . . . . O .
02 . . . . . .
03 X . . . . .
04 . . . . X .
05 . . . . . .
------------------------------
Step 4: Player 2 plays (np.int64(2), np.int64(5))
   00 01 02 03 04 05
00 . . . . . .
01 . . . . O .
02 . . . . . O
03 X . . . . .
04 . . . . X .
05 . . . . . .
------------------------------
Step 5: Player 1 plays (np.int64(0), np.int64(0))
   00

# The Core Node Structure

In [5]:
import numpy as np

class MCTSNode:
    def __init__(self, state, parent=None, move_from_parent=None):
        """
        A node in the Monte Carlo Tree.
        Works for both Hex and Gomoku as long as they share the same API.
        """
        self.state = state
        self.parent = parent
        self.move_from_parent = move_from_parent
        self.children = {}  # Format: {move: MCTSNode}
        
        # Core statistics used by both UCB and Thompson Sampling
        self.visit_count = 0
        self.win_count = 0  
        
        # Unexplored actions from this specific state
        self.untried_moves = state.get_legal_moves()
        
    def is_fully_expanded(self):
        """A node is fully expanded if all legal moves have been explored."""
        return len(self.untried_moves) == 0
        
    def is_terminal_node(self):
        """Checks if this node represents a finished game."""
        return self.state.is_terminal()
        
    def expand(self):
        """
        Pops an untried move, creates a new child state, and adds it to the tree.
        """
        # Take the last move from the untried list (O(1) operation)
        move = self.untried_moves.pop()
        
        # Generate the new state using your immutable make_move function
        next_state = self.state.make_move(move)
        
        # Create and link the new child node
        child_node = MCTSNode(next_state, parent=self, move_from_parent=move)
        self.children[move] = child_node
        
        return child_node
        
    def backpropagate(self, result):
        """
        Updates this node's statistics and recursively passes the result up the tree.
        
        result: 1 (Player 1 wins), 2 (Player 2 wins), or None (Draw)
        """
        self.visit_count += 1
        
        # MCTS perspective tracking: 
        # 'self.state.turn' tells us who is ABOUT to move. 
        # Therefore, the player who made the move to GET to this node is (3 - self.state.turn).
        player_who_just_moved = 3 - self.state.turn
        
        if result == player_who_just_moved:
            self.win_count += 1
        elif result is None:
            # For Gomoku draws, award a half-point
            self.win_count += 0.5 
            
        # Pass the result up to the root
        if self.parent is not None:
            self.parent.backpropagate(result)

# The Selection Algorithms

## Algorithm A: Standard UCT

In [6]:
def select_child_uct(node, exploration_constant=1.414):
    """
    Selects the best child using the Upper Confidence Bound applied to Trees (UCT).
    """
    best_score = -float('inf')
    best_child = None
    
    for child in node.children.values():
        # Exploit: The current win rate
        exploit_term = child.win_count / child.visit_count
        
        # Explore: Favors nodes with fewer visits
        explore_term = exploration_constant * np.sqrt(np.log(node.visit_count) / child.visit_count)
        
        ucb_score = exploit_term + explore_term
        
        if ucb_score > best_score:
            best_score = ucb_score
            best_child = child
            
    return best_child

## Thompson Sampling MCTS

In [7]:
def select_child_thompson(node, prior_alpha=1.0, prior_beta=1.0):
    """
    Selects the best child using Thompson Sampling.
    Models the win probability of each child as a Beta distribution.
    """
    best_sample = -float('inf')
    best_child = None
    
    for child in node.children.values():
        # Calculate the posterior parameters based on the node's history
        # alpha = prior + wins
        # beta = prior + losses (visits - wins)
        alpha = prior_alpha + child.win_count
        beta = prior_beta + (child.visit_count - child.win_count)
        
        # Draw a sample from the Beta distribution
        sampled_value = np.random.beta(alpha, beta)
        
        if sampled_value > best_sample:
            best_sample = sampled_value
            best_child = child
            
    return best_child

# The Rollout Policy & Main Loop

In [8]:
import random
import time

def random_rollout(state):
    """
    Phase 3: Simulation.
    Plays random moves from the current state until a terminal condition is met.
    Returns the winner (1 or 2, or None for a draw).
    """
    current_state = state
    while not current_state.is_terminal():
        legal_moves = current_state.get_legal_moves()
        # Fast random choice
        move = random.choice(legal_moves)
        current_state = current_state.make_move(move)
    return current_state.check_winner()


def run_mcts(root_state, iterations, selection_strategy, **strategy_kwargs):
    """
    Executes the MCTS algorithm for a fixed number of iterations.
    
    selection_strategy: Either select_child_uct or select_child_thompson
    strategy_kwargs: Hyperparameters like exploration_constant or priors
    """
    root_node = MCTSNode(state=root_state)

    for _ in range(iterations):
        node = root_node

        # Phase 1: Selection
        # Traverse down the tree using the chosen selection strategy 
        # until we hit a node that isn't fully expanded or is terminal.
        while node.is_fully_expanded() and not node.is_terminal_node():
            node = selection_strategy(node, **strategy_kwargs)

        # Phase 2: Expansion
        # If the node is not terminal, expand it by creating one new child.
        if not node.is_terminal_node():
            node = node.expand()

        # Phase 3: Simulation (Rollout)
        # Play out the game randomly from this new node state.
        result = random_rollout(node.state)

        # Phase 4: Backpropagation
        # Feed the result back up to the root node.
        node.backpropagate(result)

    # After finishing iterations, choose the final move.
    # In MCTS, the best move is the one belonging to the most visited child.
    best_move = max(root_node.children.items(), key=lambda item: item[1].visit_count)[0]
    
    return best_move, root_node

# Setting Up The Instrumentation (Metrics Collection)

In [9]:
def calculate_root_entropy(root_node):
    """
    Calculates the Shannon Entropy of the root node's child visit distribution.
    Higher entropy = Thompson Sampling explored more broadly / diversely.
    Lower entropy = UCT focused heavily on a single path.
    """
    if not root_node.children:
        return 0.0
        
    total_visits = sum(child.visit_count for child in root_node.children.values())
    if total_visits == 0:
        return 0.0
        
    entropy = 0.0
    for child in root_node.children.values():
        if child.visit_count > 0:
            p_i = child.visit_count / total_visits
            entropy -= p_i * np.log(p_i)
            
    return entropy

# The Ultimate Test Drive

In [11]:
# Assuming your previous code blocks are imported or in the same file:
if __name__ == "__main__":
    # Initialize a small Hex board for an instant sanity check
    game_state = HexState(board_size=4)
    print("--- Starting AI vs AI Test Match ---")
    print(game_state)
    print("-" * 40)
    
    while not game_state.is_terminal():
        start_time = time.time()
        
        if game_state.turn == 1:
            # Player 1 uses UCT
            print("UCT (Player 1) is thinking...")
            move, root = run_mcts(game_state, iterations=400, selection_strategy=select_child_uct, exploration_constant=1.4)
            strategy_used = "UCT"
        else:
            # Player 2 uses Thompson Sampling
            print("Thompson Sampling (Player 2) is thinking...")
            move, root = run_mcts(game_state, iterations=400, selection_strategy=select_child_thompson, prior_alpha=1.0, prior_beta=1.0)
            strategy_used = "Thompson"
            
        elapsed = time.time() - start_time
        entropy = calculate_root_entropy(root)
        
        # Execute move
        game_state = game_state.make_move(move)
        
        print(f"[{strategy_used}] Selected Move: {move} | Time: {elapsed:.3f}s | Root Entropy: {entropy:.4f}")
        print(game_state)
        print("-" * 40)
        
    print(f"Game Over! Winner is Player {game_state.check_winner()}")

--- Starting AI vs AI Test Match ---
. . . .
 . . . .
  . . . .
   . . . .
----------------------------------------
UCT (Player 1) is thinking...
[UCT] Selected Move: (np.int64(0), np.int64(3)) | Time: 0.159s | Root Entropy: 2.7293
. . . X
 . . . .
  . . . .
   . . . .
----------------------------------------
Thompson Sampling (Player 2) is thinking...
[Thompson] Selected Move: (np.int64(1), np.int64(2)) | Time: 0.161s | Root Entropy: 2.4808
. . . X
 . . O .
  . . . .
   . . . .
----------------------------------------
UCT (Player 1) is thinking...
[UCT] Selected Move: (np.int64(1), np.int64(3)) | Time: 0.139s | Root Entropy: 2.5701
. . . X
 . . O X
  . . . .
   . . . .
----------------------------------------
Thompson Sampling (Player 2) is thinking...
[Thompson] Selected Move: (np.int64(2), np.int64(2)) | Time: 0.128s | Root Entropy: 2.2291
. . . X
 . . O X
  . . O .
   . . . .
----------------------------------------
UCT (Player 1) is thinking...
[UCT] Selected Move: (np.int64(2), n